In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from jax.random import PRNGKey
from jax import numpy as jnp

pd.options.plotting.backend = "plotly"

from summer3.graph import *
from summer3.epi import Stratification, CompartmentMap, \
    CategoryGroup, StratSpec, mixing_matrix, ManagedArray, \
    CategoryData, TransitionFlow, CompartmentalEpiModel, \
    CompartmentalModelODE, strat_data_from_pandas, \
    build_istate, dti_to_epoch

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)

In [ ]:
# Infection process
def new_iprocess(compartment_values: ManagedArray, contact_rate: float, infectious_compartments):
    return contact_rate * compartment_values.query(infectious_compartments).sum() / compartment_values.sum()

foi = defer(new_iprocess)(
    CompartmentValues, Parameter("contact_rate", 0.0), disease_state["I"],
)
infection = TransitionFlow(
    "infection", 
    disease_state["S"], 
    disease_state["I"], 
    foi,
)
epi_model.add_flow(infection)

# Recovery process
recovery = TransitionFlow(
    "recovery",
    disease_state["I"],
    disease_state["R"],
    1.0 / Parameter("recovery_time", 0.0),
)
epi_model.add_flow(recovery)

In [ ]:
epi_model.set_initial_population(
    CategoryData(disease_state.categories(), np.array([999.0, 1.0, 0.0]))
)

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(len(epi_model.times), dti_to_epoch(epi_model.times), True)
    return runner, istate

In [ ]:
params = {"contact_rate": 0.2, "recovery_time": 20.0}
runner, istate = get_runner(epi_model)
results = epi_model.run(params)

In [ ]:
results["compartments"].to_pandas_df().plot()